<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/ml/notebooks/c5_l7.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C5-L7 · Costos: 2 bps + slippage
Del bruto al neto: comisión fija más slippage por rotación, y un umbral de convicción para dejar de donarle al mercado.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/ml/data/c5_l7.csv'
try:
    df = pd.read_csv(URL)
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c5_l7.csv'), Path('data/c5_l7.csv'), Path('c5_l7.csv')]:
        if cand.exists():
            df = pd.read_csv(cand); break
    print('Fuente: local')
print(df.shape)
print(df.head(5).to_string(index=False))

In [ ]:
df['ret'] = df['close'].pct_change()
df['rango'] = (df['high']-df['low'])/df['close']
for k in (1, 2, 3, 5):
    df[f'lag_{k}'] = df['ret'].shift(k)
df['mom5'] = df['close']/df['close'].shift(5) - 1
data = df.dropna().reset_index(drop=True)
data['fwd'] = data['ret'].shift(-1)
data = data.iloc[:-1].reset_index(drop=True)
COMISION = 0.0002  # 2 bps
data['slip'] = 0.25*data['rango']  # cruzar el spread ≈ 1/4 del rango
data['pos'] = np.where(data['mom5'] > 0, 1.0, -1.0)
print('filas:', len(data))
assert len(data) > 40 and data[['fwd', 'pos']].isna().sum().sum() == 0

In [ ]:
def sharpe(x):
    x = np.asarray(x, float)
    return float(x.mean()/x.std()*np.sqrt(252)) if x.std() > 0 else 0.0
pos = data['pos'].values; fwd = data['fwd'].values
turnover = np.abs(np.diff(np.r_[0, pos]))/2
costo = turnover*(COMISION + data['slip'].values)
pnl_bruto = pos*fwd
pnl_neto = pos*fwd - costo
sh_b, sh_n = sharpe(pnl_bruto), sharpe(pnl_neto)
print(f'bruto={sh_b:.3f} neto={sh_n:.3f} trades={int(turnover.sum())} costo_total={float(costo.sum()):.5f}')

In [ ]:
UMBRAL = 0.006  # deadband: solo un giro fuerte cambia la posición, si no se mantiene
mom = data['mom5'].values
pos_f = np.zeros_like(pos)
pos_f[0] = pos[0]
for t in range(1, len(mom)):
    pos_f[t] = np.sign(mom[t]) if abs(mom[t]) > UMBRAL else pos_f[t-1]
to_f = np.abs(np.diff(np.r_[0, pos_f]))/2
pnl_f = pos_f*fwd - to_f*(COMISION + data['slip'].values)
sh_f = sharpe(pnl_f)
print(f'filtrado: neto={sh_f:.3f} trades={int(to_f.sum())} (antes {int(turnover.sum())})')

In [ ]:
assert np.isfinite([sh_b, sh_n, sh_f]).all()
assert float(costo.sum()) > 0
assert pnl_neto.sum() <= pnl_bruto.sum() + 1e-12
assert int(to_f.sum()) < int(turnover.sum())  # el deadband recorta rotación
assert sh_f > sh_n  # y el neto mejora operando solo giros fuertes
print(f'OK L7: bruto {sh_b:.3f} -> neto {sh_n:.3f} -> filtrado {sh_f:.3f} ({int(turnover.sum())}->{int(to_f.sum())} trades)')